In [1]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder,
    FewShotChatMessagePromptTemplate,
)

from langchain_core.messages import (
    HumanMessage,
    AIMessage,
)


# ============================================================
# 1. Few-Shot Examples
# ============================================================

examples = [
    {
        "question": "چطور رمز عبورم را تغییر بدهم؟",
        "answer": "از بخش تنظیمات حساب کاربری، گزینه تغییر رمز عبور را انتخاب کنید."
    },
    {
        "question": "چطور درخواست مرخصی ثبت کنم؟",
        "answer": "از سامانه منابع انسانی وارد بخش درخواست مرخصی شوید."
    }
]


# ============================================================
# 2. Template مربوط به هر Example
# ============================================================

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{question}"),
    ("ai", "{answer}")
])


# ============================================================
# 3. ساخت Few-Shot Prompt
# ============================================================

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)


# ============================================================
# 4. ساخت Prompt اصلی
# ============================================================

prompt = ChatPromptTemplate.from_messages([

    # -------------------------
    # System Prompt
    # -------------------------
    (
        "system",
        """
تو یک دستیار هوش مصنوعی سازمانی هستی.

قوانین:
- پاسخ‌ها باید دقیق و کوتاه باشند.
- اگر اطلاعات کافی نداری، صریحاً بگو اطلاعات کافی ندارم.
- اطلاعات محرمانه را افشا نکن.
- سطح پاسخ را متناسب با نقش کاربر تنظیم کن.

نقش کاربر:
{user_role}
"""
    ),

    # -------------------------
    # Few-Shot Examples
    # -------------------------
    few_shot_prompt,

    # -------------------------
    # Conversation History
    # -------------------------
    MessagesPlaceholder(
        variable_name="history"
    ),

    # -------------------------
    # Current User Question
    # -------------------------
    (
        "human",
        "{question}"
    )
])


# ============================================================
# 5. Conversation History
# ============================================================

history = [
    HumanMessage(
        "گزارش فروش شعبه را می‌خواهم."
    ),

    AIMessage(
        "برای کدام بازه زمانی؟"
    ),

    HumanMessage(
        "ماه گذشته."
    )
]


# ============================================================
# 6. مقداردهی Template
# ============================================================

result = prompt.invoke({

    "user_role": "مدیر شعبه",

    "history": history,

    "question": "حالا فقط فروش محصولات اعتباری را نشان بده."
})


# ============================================================
# 7. مشاهده Messageهای نهایی
# ============================================================

for message in result.messages:

    print("=" * 60)

    print("TYPE:")
    print(type(message).__name__)

    print("\nCONTENT:")
    print(message.content)

    print()

TYPE:
SystemMessage

CONTENT:

تو یک دستیار هوش مصنوعی سازمانی هستی.

قوانین:
- پاسخ‌ها باید دقیق و کوتاه باشند.
- اگر اطلاعات کافی نداری، صریحاً بگو اطلاعات کافی ندارم.
- اطلاعات محرمانه را افشا نکن.
- سطح پاسخ را متناسب با نقش کاربر تنظیم کن.

نقش کاربر:
مدیر شعبه


TYPE:
HumanMessage

CONTENT:
چطور رمز عبورم را تغییر بدهم؟

TYPE:
AIMessage

CONTENT:
از بخش تنظیمات حساب کاربری، گزینه تغییر رمز عبور را انتخاب کنید.

TYPE:
HumanMessage

CONTENT:
چطور درخواست مرخصی ثبت کنم؟

TYPE:
AIMessage

CONTENT:
از سامانه منابع انسانی وارد بخش درخواست مرخصی شوید.

TYPE:
HumanMessage

CONTENT:
گزارش فروش شعبه را می‌خواهم.

TYPE:
AIMessage

CONTENT:
برای کدام بازه زمانی؟

TYPE:
HumanMessage

CONTENT:
ماه گذشته.

TYPE:
HumanMessage

CONTENT:
حالا فقط فروش محصولات اعتباری را نشان بده.



In [5]:
from langchain.chat_models import init_chat_model


model = init_chat_model(
    "qwen3:1.7b",
    model_provider="ollama",
    temperature=0
)


chain = prompt | model


response = chain.invoke({

    "user_role": "مدیر شعبه",

    "history": history,

    "question": "حالا فقط فروش محصولات اعتباری را نشان بده."
})
print(response.content)

برای درخواست گزارش فروش، از سامانه منابع انسانی وارد بخش درخواست گزارش فروش شوید و اطلاعات مورد نظر را اعلام کنید.
